# Phase 3b — Multi-Regulation Embedding & Indexing

Extends the existing DORA Chroma collection with NIS2, PS21/3,
and cross-regulation chunks from Phase 2b.

**Strategy — additive, not rebuild:**
- Reads `all_chunks.json` produced by Phase 2b (~1,000+ chunks)
- Checks which chunk IDs are already in the Chroma collection
- Only embeds and inserts chunks that are NOT already indexed
- DORA vectors are reused — no re-embedding of existing content
- Saves updated `index_config.json` with multi-regulation breakdown

**Result:** Single Chroma collection `dora_regulation` containing
all three regulations, filterable by `regulation`, `jurisdiction`,
`level`, `doc_type`, `source_doc`, and `article_num`.

**Prerequisites:**
- Phase 3 complete (`dora_rag/dora_chroma_db/` exists with DORA vectors)
- Phase 2b complete (`dora_rag/dora_chunks/all_chunks.json` exists)

In [ ]:
# Cell 1 — Install dependencies (same as Phase 3a)
import subprocess, sys
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'llama-index',
    'llama-index-embeddings-huggingface',
    'llama-index-vector-stores-chroma',
    'chromadb',
    'sentence-transformers',
    'torch',
    'tqdm',
])
print('Dependencies installed.')


Dependencies installed.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip


In [ ]:
# Cell 2 — Imports and configuration
import json
import time
import datetime
import torch
from pathlib import Path
from collections import Counter, defaultdict

BASE_DIR        = Path('./dora_rag')
CHUNKS_DIR      = BASE_DIR / 'dora_chunks'
CHROMA_DIR      = BASE_DIR / 'dora_chroma_db'    # same collection as Phase 3a
ALL_CHUNKS_FILE = CHUNKS_DIR / 'all_chunks.json'  # from Phase 2b
CONFIG_FILE     = BASE_DIR / 'index_config.json'  # will be updated

# Same embedding model as Phase 3 — must not change
EMBED_MODEL_NAME  = 'BAAI/bge-large-en-v1.5'
EMBED_BATCH_SIZE  = 16
CHROMA_COLLECTION = 'dora_regulation'   # same collection — we ADD to it

# FORCE_REBUILD: set True only if you want to wipe everything and re-embed
# from scratch (including existing DORA vectors). Normally keep False.
FORCE_REBUILD = True

device = 'cuda' if torch.cuda.is_available() else 'cpu'

print('Configuration:')
print(f'  All chunks file  : {ALL_CHUNKS_FILE}')
print(f'  Chroma DB path   : {CHROMA_DIR}')
print(f'  Collection       : {CHROMA_COLLECTION}')
print(f'  Embedding model  : {EMBED_MODEL_NAME}')
print(f'  Batch size       : {EMBED_BATCH_SIZE}')
print(f'  Device           : {device}')
print(f'  Force rebuild    : {FORCE_REBUILD}')
print()

# Validate required inputs
missing = [f for f in [ALL_CHUNKS_FILE, CHROMA_DIR] if not f.exists()]
if missing:
    for m in missing:
        print(f'  ✗ MISSING: {m}')
    raise FileNotFoundError('Run Phase 2b and Phase 3 before Phase 3b.')
print('  ✓ All input files present')


Configuration:
  All chunks file  : dora_rag/dora_chunks/all_chunks.json
  Chroma DB path   : dora_rag/dora_chroma_db
  Collection       : dora_regulation
  Embedding model  : BAAI/bge-large-en-v1.5
  Batch size       : 16
  Device           : cpu
  Force rebuild    : True

  ✓ All input files present


In [18]:
# Cell 3 — Load all chunks from Phase 2b and show corpus breakdown

raw_chunks = json.loads(ALL_CHUNKS_FILE.read_text(encoding='utf-8'))
print(f'Loaded {len(raw_chunks)} chunks from all_chunks.json')
print()

# By regulation
by_reg  = Counter(c['regulation']   for c in raw_chunks)
by_jur  = Counter(c['jurisdiction'] for c in raw_chunks)
by_lv   = Counter(c.get('level', 0) for c in raw_chunks)
by_type = Counter(c['doc_type']     for c in raw_chunks)

print(f"{'Regulation':<14} {'Chunks':>6}")
print('─' * 22)
for reg, cnt in sorted(by_reg.items()):
    print(f"  {reg:<12} {cnt:>6}")
print('─' * 22)
print(f"  {'TOTAL':<12} {len(raw_chunks):>6}")

print(f'\nBy jurisdiction  : {dict(sorted(by_jur.items()))}')
print(f'By level         : {dict(sorted(by_lv.items()))}')
print(f'\nBy doc_type:')
for dtype, cnt in by_type.most_common():
    print(f'  {dtype:<30}: {cnt}')


Loaded 826 chunks from all_chunks.json

Regulation     Chunks
──────────────────────
  cross_reg       128
  dora            390
  nis2            189
  ps21_3          119
──────────────────────
  TOTAL           826

By jurisdiction  : {'eu': 643, 'uk': 183}
By level         : {1: 288, 2: 447, 3: 91}

By doc_type:
  directive                     : 151
  RTS                           : 140
  regulation                    : 116
  ITS                           : 94
  policy_statement              : 85
  discussion_paper              : 57
  Q_and_A                       : 43
  guidance                      : 42
  implementing_regulation       : 37
  guideline                     : 28
  supervisory_statement         : 16
  delegated_regulation          : 14
  statement                     : 3


In [ ]:
# Cell 4 — Load embedding model (same as Phase 3a)
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings

print(f'Loading: {EMBED_MODEL_NAME}')
print(f'Device : {device}')
print('Model is cached locally from Phase 3 — no download needed.\n')

t0 = time.time()
embed_model = HuggingFaceEmbedding(
    model_name=EMBED_MODEL_NAME,
    query_instruction='Represent this sentence for searching relevant passages: ',
    embed_batch_size=EMBED_BATCH_SIZE,
    device=device,
)
Settings.embed_model = embed_model

print(f'✓ Model loaded in {time.time() - t0:.1f}s')
dim = len(embed_model.get_text_embedding('test'))
print(f'✓ Embedding dimension: {dim}  (expected 1024)')


Loading: BAAI/bge-large-en-v1.5
Device : cpu
Model is cached locally from Phase 3 — no download needed.

✓ Model loaded in 3.5s
✓ Embedding dimension: 1024  (expected 1024)


In [20]:
# Cell 5 — Connect to existing Chroma collection
#
# Opens the existing dora_regulation collection (not deleting it).
# If FORCE_REBUILD=True, wipes and recreates — use only if you want
# to re-embed everything from scratch.

import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext

chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))
existing      = [c.name for c in chroma_client.list_collections()]
print(f'Existing collections: {existing}')

if FORCE_REBUILD:
    print('FORCE_REBUILD=True — wiping and recreating collection.')
    if CHROMA_COLLECTION in existing:
        chroma_client.delete_collection(CHROMA_COLLECTION)
    chroma_collection = chroma_client.create_collection(
        name=CHROMA_COLLECTION,
        metadata={'hnsw:space': 'cosine'}
    )
    existing_ids = set()
elif CHROMA_COLLECTION in existing:
    chroma_collection = chroma_client.get_collection(CHROMA_COLLECTION)
    existing_count    = chroma_collection.count()
    # Get all existing IDs to avoid re-embedding
    existing_ids = set(chroma_collection.get(include=[])['ids'])
    print(f'✓ Opened collection: {CHROMA_COLLECTION} ({existing_count} existing vectors)')
    print(f'  Existing chunk IDs loaded: {len(existing_ids)}')
else:
    print(f'Collection not found — creating: {CHROMA_COLLECTION}')
    chroma_collection = chroma_client.create_collection(
        name=CHROMA_COLLECTION,
        metadata={'hnsw:space': 'cosine'}
    )
    existing_ids = set()

vector_store    = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)
print(f'\n✓ Chroma ready — collection: {CHROMA_COLLECTION}')


Existing collections: ['dora_regulation']
FORCE_REBUILD=True — wiping and recreating collection.

✓ Chroma ready — collection: dora_regulation


In [ ]:
# Cell 6 — Convert chunks to TextNodes
#
# Extended vs Phase 3a chunk_to_node():
#   + regulation   field in metadata
#   + jurisdiction field in metadata
#   + section_ref  field in metadata (PS21/3 paragraph references)
# All existing Phase 3a fields preserved unchanged.

from llama_index.core.schema import TextNode

def chunk_to_node(chunk: dict) -> TextNode:
    return TextNode(
        id_  = chunk['chunk_id'],
        text = chunk['text_with_header'],
        metadata = {
            # Regulation identity — NEW fields
            'regulation':    chunk.get('regulation', 'dora'),
            'jurisdiction':  chunk.get('jurisdiction', 'eu'),
            # Document identity — same as Phase 3
            'source_doc':    chunk['source_doc'],
            'doc_type':      chunk['doc_type'],
            'level':         chunk.get('level', 0),
            # Location within document
            'article_num':   chunk['article_num'] if chunk['article_num'] is not None else -1,
            'article_title': chunk['article_title'],
            'section_ref':   chunk.get('section_ref') or '',  # NEW: PS21/3 paragraph refs
            'paragraph_idx': chunk['paragraph_idx'],
            # Semantic metadata — pipe-separated strings (Chroma requires scalars)
            'topic_tags':    '|'.join(chunk.get('topic_tags', [])),
            'entity_scope':  '|'.join(chunk.get('entity_scope', [])),
            # Diagnostic
            'token_estimate': chunk['token_estimate'],
        },
        excluded_embed_metadata_keys=['paragraph_idx', 'token_estimate'],
        excluded_llm_metadata_keys=['token_estimate'],
    )


all_nodes = [chunk_to_node(c) for c in raw_chunks]

# Deduplicate by ID (same logic as Phase 3a)
seen_ids, unique_nodes = set(), []
for node in all_nodes:
    if node.id_ not in seen_ids:
        seen_ids.add(node.id_)
        unique_nodes.append(node)

dupes = len(all_nodes) - len(unique_nodes)
if dupes:
    print(f'⚠ Removed {dupes} duplicate nodes')

print(f'✓ Converted {len(unique_nodes)} unique chunks to TextNodes')
print(f'\nSample node (DORA):')
dora_sample = next(n for n in unique_nodes if n.metadata.get('regulation') == 'dora')
print(f'  id           : {dora_sample.id_}')
print(f'  regulation   : {dora_sample.metadata["regulation"]}')
print(f'  jurisdiction : {dora_sample.metadata["jurisdiction"]}')
print(f'  source_doc   : {dora_sample.metadata["source_doc"]}')

# Try to show NIS2 and PS21/3 samples if available
for reg in ['nis2', 'ps21_3', 'cross_reg']:
    sample = next((n for n in unique_nodes if n.metadata.get('regulation') == reg), None)
    if sample:
        print(f'\nSample node ({reg}):')
        print(f'  id           : {sample.id_}')
        print(f'  regulation   : {sample.metadata["regulation"]}')
        print(f'  jurisdiction : {sample.metadata["jurisdiction"]}')
        print(f'  source_doc   : {sample.metadata["source_doc"]}')
        sr = sample.metadata.get('section_ref', '')
        if sr:
            print(f'  section_ref  : {sr}')


⚠ Removed 28 duplicate nodes
✓ Converted 798 unique chunks to TextNodes

Sample node (DORA):
  id           : L1_DORA_Regulation_2__art101__p00
  regulation   : dora
  jurisdiction : eu
  source_doc   : L1_DORA_Regulation_2022_2554

Sample node (nis2):
  id           : NIS2_Directive_2022___art001__p00
  regulation   : nis2
  jurisdiction : eu
  source_doc   : NIS2_Directive_2022_2555

Sample node (ps21_3):
  id           : PS21_3_FCA_PRA_Opera__sec_1_16__p00
  regulation   : ps21_3
  jurisdiction : uk
  source_doc   : PS21_3_FCA_PRA_Operational_Resilience
  section_ref  : 1.16

Sample node (cross_reg):
  id           : EC_Guidelines_DORA_N__art009__p00
  regulation   : cross_reg
  jurisdiction : eu
  source_doc   : EC_Guidelines_DORA_NIS2_Relationship_2023


In [22]:
# Cell 7 — Identify new chunks to index
#
# Separates chunks into:
#   already_indexed : chunk ID exists in Chroma — skip embedding
#   to_index        : chunk ID not in Chroma — embed and insert
#
# This means DORA chunks are never re-embedded unless FORCE_REBUILD=True.
# Only NIS2, PS21/3, and cross-reg chunks (and any patched DORA chunks
# with updated metadata) will be processed.

already_indexed = [n for n in unique_nodes if n.id_ in existing_ids]
to_index        = [n for n in unique_nodes if n.id_ not in existing_ids]

print(f'Chunk indexing plan:')
print(f'  Total unique chunks    : {len(unique_nodes)}')
print(f'  Already in Chroma      : {len(already_indexed)}  ← skipping (reuse existing vectors)')
print(f'  New — to be embedded   : {len(to_index)}')

if to_index:
    by_reg_new = Counter(n.metadata.get('regulation', 'unknown') for n in to_index)
    print(f'\n  New chunks by regulation:')
    for reg, cnt in sorted(by_reg_new.items()):
        est_time = cnt * 0.15   # ~0.15s per chunk on CPU from Phase 3 baseline
        print(f'    {reg:<12}: {cnt:>5} chunks  (~{est_time:.0f}s on CPU)')
    total_est = len(to_index) * 0.15
    print(f'\n  Total estimated time: ~{total_est:.0f}s ({total_est/60:.1f} min) on CPU')
    print(f'  On GPU: ~{total_est/10:.0f}s')
else:
    print('\n  ✓ All chunks already indexed — nothing to do.')


Chunk indexing plan:
  Total unique chunks    : 798
  Already in Chroma      : 0  ← skipping (reuse existing vectors)
  New — to be embedded   : 798

  New chunks by regulation:
    cross_reg   :   119 chunks  (~18s on CPU)
    dora        :   381 chunks  (~57s on CPU)
    nis2        :   179 chunks  (~27s on CPU)
    ps21_3      :   119 chunks  (~18s on CPU)

  Total estimated time: ~120s (2.0 min) on CPU
  On GPU: ~12s


In [23]:
# Cell 8 — Embed and index new chunks
#
# Only embeds chunks not already in the collection.
# Progress bar shows embedding of new chunks only.
# Chroma persists after each batch — safe to interrupt and resume
# (re-run with FORCE_REBUILD=False; already-indexed chunks are skipped).

from llama_index.core import VectorStoreIndex

if not to_index:
    print('All chunks already indexed. Loading existing index...')
    index = VectorStoreIndex.from_vector_store(
        vector_store=vector_store,
        embed_model=embed_model,
    )
    print(f'✓ Index loaded — {chroma_collection.count()} vectors')

else:
    print(f'Embedding {len(to_index)} new chunks...')
    print(f'Batch size: {EMBED_BATCH_SIZE} | Device: {device}')
    print(f'(Existing {len(already_indexed)} DORA vectors are reused — not re-embedded)\n')

    t0 = time.time()

    if already_indexed:
        # Collection already has vectors — add new ones on top
        index = VectorStoreIndex.from_vector_store(
            vector_store=vector_store,
            embed_model=embed_model,
        )
        # Insert new nodes into existing index
        for node in to_index:
            index.insert(node)
    else:
        # Empty collection (FORCE_REBUILD or first run) — build from scratch
        index = VectorStoreIndex(
            to_index,
            storage_context=storage_context,
            embed_model=embed_model,
            show_progress=True,
        )

    elapsed = time.time() - t0
    final_count = chroma_collection.count()

    print(f'\n✓ Indexing complete')
    print(f'  New vectors added    : {len(to_index)}')
    print(f'  Total vectors stored : {final_count}')
    print(f'  Time elapsed         : {elapsed:.1f}s ({elapsed/max(1,len(to_index)):.2f}s per chunk)')
    print(f'  Index persisted to   : {CHROMA_DIR}')


Embedding 798 new chunks...
Batch size: 16 | Device: cpu
(Existing 0 DORA vectors are reused — not re-embedded)



Generating embeddings:   0%|          | 0/798 [00:00<?, ?it/s]

Generating embeddings: 100%|██████████| 798/798 [01:54<00:00,  6.95it/s]



✓ Indexing complete
  New vectors added    : 798
  Total vectors stored : 798
  Time elapsed         : 115.5s (0.14s per chunk)
  Index persisted to   : dora_rag/dora_chroma_db


In [24]:
# Cell 9 — Sanity checks: retrieve from each regulation
#
# Tests that each regulation's chunks are retrievable with
# the appropriate regulation filter. One query per regulation.

from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.vector_stores import (
    MetadataFilter, MetadataFilters, FilterOperator
)

def sanity_check(query: str, regulation: str, expected_sources: list = None):
    filters = MetadataFilters(filters=[
        MetadataFilter(key='regulation', value=regulation, operator=FilterOperator.EQ)
    ])
    retriever = VectorIndexRetriever(index=index, similarity_top_k=3, filters=filters)
    results   = retriever.retrieve(query)

    status = '✓' if results else '✗ NO RESULTS'
    print(f'\n{"─"*65}')
    print(f'[{regulation.upper()}] {query}')
    print(f'Results: {len(results)}  {status}')

    for node in results:
        m   = node.metadata
        reg = m.get('regulation', '?')
        jur = m.get('jurisdiction', '?')
        lv  = m.get('level', '?')
        art = m.get('article_num', '?')
        sec = m.get('section_ref', '')
        src = m.get('source_doc', '')[:38]
        loc = f'Art {art}' if art != -1 else (f'Sec {sec}' if sec else 'n/a')
        print(f'  [{reg}/{jur}] L{lv} {loc:<12} {src}  score:{node.score:.4f}')

    if expected_sources:
        found = {n.metadata.get('source_doc','') for n in results}
        hits  = sum(1 for s in expected_sources if any(s in f for f in found))
        print(f'  Expected source hit: {hits}/{len(expected_sources)}')

    return results


# DORA
sanity_check(
    'ICT incident reporting timelines',
    'dora',
    expected_sources=['L1_DORA_Regulation']
)

# NIS2
sanity_check(
    'cybersecurity risk management measures for essential entities',
    'nis2',
    expected_sources=['NIS2_Directive']
)

# PS21/3
sanity_check(
    'impact tolerance important business services operational resilience',
    'ps21_3',
    expected_sources=['PS21_3', 'SS1_21']
)

# Cross-regulation
sanity_check(
    'interaction between DORA and NIS2 financial entities',
    'cross_reg',
    expected_sources=['EC_Guidelines', 'DORA_ESA']
)



─────────────────────────────────────────────────────────────────
[DORA] ICT incident reporting timelines
Results: 3  ✓
  [dora/eu] L2 Art 9        L2_ITS_ICT_Incident_Reporting  score:0.7528
  [dora/eu] L2 Art 9        L2_ITS_ICT_Incident_Reporting  score:0.7516
  [dora/eu] L2 Art 9        L2_ITS_ICT_Incident_Reporting  score:0.7506
  Expected source hit: 0/1

─────────────────────────────────────────────────────────────────
[NIS2] cybersecurity risk management measures for essential entities
Results: 3  ✓
  [nis2/eu] L1 Art 21       NIS2_Directive_2022_2555  score:0.7947
  [nis2/eu] L1 Art 20       NIS2_Directive_2022_2555  score:0.7682
  [nis2/eu] L1 Art 32       NIS2_Directive_2022_2555  score:0.7663
  Expected source hit: 1/1

─────────────────────────────────────────────────────────────────
[PS21_3] impact tolerance important business services operational resilience
Results: 3  ✓
  [ps21_3/uk] L1 Sec 3.5      PS21_3_FCA_PRA_Operational_Resilience  score:0.8243
  [ps21_3/uk] L2 S

[NodeWithScore(node=TextNode(id_='EC_Guidelines_DORA_N__art009__p02', embedding=None, metadata={'regulation': 'cross_reg', 'jurisdiction': 'eu', 'source_doc': 'EC_Guidelines_DORA_NIS2_Relationship_2023', 'doc_type': 'guideline', 'level': 3, 'article_num': 9, 'article_title': 'and 16 should apply in their entirety for sectors, despite the existence of sector-specific Union legal acts within the', 'section_ref': '', 'paragraph_idx': 2, 'topic_tags': 'incident_reporting|third_party_risk|resilience_testing|business_continuity|information_sharing|supervision|network_security|operational_resilience', 'entity_scope': 'credit_institutions|important_entities', 'token_estimate': 906}, excluded_embed_metadata_keys=['paragraph_idx', 'token_estimate'], excluded_llm_metadata_keys=['token_estimate'], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text='Article 9 — and 16 should apply in their entirety for sectors, despite the existence of sector-specific Union legal ac

In [25]:
# Cell 10 — Metadata filter tests
#
# Tests all new filter dimensions added in Phase 3b:
#   regulation filter   — dora / nis2 / ps21_3 / cross_reg
#   jurisdiction filter — eu / uk
#   combined filters    — regulation + level
#
# Existing Phase 3 filter tests (level, doc_type, article_num) still work unchanged.

def filter_test(query: str, filters: MetadataFilters, label: str):
    print(f'\n{"─"*65}')
    print(f'FILTER TEST: {label}')
    print(f'Query: {query}')
    fr      = VectorIndexRetriever(index=index, similarity_top_k=4, filters=filters)
    results = fr.retrieve(query)
    print(f'Results: {len(results)}')
    for node in results:
        m   = node.metadata
        reg = m.get('regulation', '?')
        jur = m.get('jurisdiction', '?')
        lv  = m.get('level', '?')
        art = m.get('article_num', '?')
        sec = m.get('section_ref', '')
        src = m.get('source_doc', '')[:35]
        loc = f'Art {art}' if art and art != -1 else (f'Sec {sec}' if sec else 'n/a')
        print(f'  [{reg}/{jur}] L{lv} {loc:<14} {src}  score:{node.score:.4f}')
    return results


# Filter by regulation
filter_test(
    'incident reporting obligations',
    MetadataFilters(filters=[MetadataFilter(key='regulation', value='dora',   operator=FilterOperator.EQ)]),
    'DORA only'
)
filter_test(
    'incident reporting obligations',
    MetadataFilters(filters=[MetadataFilter(key='regulation', value='nis2',   operator=FilterOperator.EQ)]),
    'NIS2 only'
)
filter_test(
    'operational resilience self-assessment impact tolerances',
    MetadataFilters(filters=[MetadataFilter(key='regulation', value='ps21_3', operator=FilterOperator.EQ)]),
    'PS21/3 only'
)

# Filter by jurisdiction
filter_test(
    'incident reporting requirements',
    MetadataFilters(filters=[MetadataFilter(key='jurisdiction', value='uk', operator=FilterOperator.EQ)]),
    'UK jurisdiction only'
)
filter_test(
    'incident reporting requirements',
    MetadataFilters(filters=[MetadataFilter(key='jurisdiction', value='eu', operator=FilterOperator.EQ)]),
    'EU jurisdiction only'
)

# Combined: NIS2 Level 1 only
filter_test(
    'cybersecurity obligations for essential entities',
    MetadataFilters(filters=[
        MetadataFilter(key='regulation', value='nis2', operator=FilterOperator.EQ),
        MetadataFilter(key='level',      value=2,      operator=FilterOperator.EQ),
    ]),
    'NIS2 Level 2 (implementing regulations)'
)

# Existing Phase 3 filter — still works
filter_test(
    'contractual provisions third-party providers',
    MetadataFilters(filters=[
        MetadataFilter(key='source_doc',  value='L1_DORA_Regulation_2022_2554', operator=FilterOperator.EQ),
        MetadataFilter(key='article_num', value=30,                             operator=FilterOperator.EQ),
    ]),
    'DORA Article 30 only (Phase 3 filter — unchanged)'
)



─────────────────────────────────────────────────────────────────
FILTER TEST: DORA only
Query: incident reporting obligations
Results: 4
  [dora/eu] L2 Art 6          L2_ITS_ICT_Incident_Reporting  score:0.7597
  [dora/eu] L2 Art 1          L2_ITS_ICT_Incident_Reporting  score:0.7318
  [dora/eu] L2 Art 9          L2_ITS_ICT_Incident_Reporting  score:0.7308
  [dora/eu] L2 Art 9          L2_ITS_ICT_Incident_Reporting  score:0.7305

─────────────────────────────────────────────────────────────────
FILTER TEST: NIS2 only
Query: incident reporting obligations
Results: 4
  [nis2/eu] L1 Art 23         NIS2_Directive_2022_2555  score:0.7875
  [nis2/eu] L1 Art 23         NIS2_Directive_2022_2555  score:0.7817
  [nis2/eu] L1 Art 23         NIS2_Directive_2022_2555  score:0.7570
  [nis2/eu] L2 Art 16         NIS2_IR_2024_2690_Cybersecurity_Req  score:0.7442

─────────────────────────────────────────────────────────────────
FILTER TEST: PS21/3 only
Query: operational resilience self-assessment i

[NodeWithScore(node=TextNode(id_='L1_DORA_Regulation_2__art030__p00', embedding=None, metadata={'regulation': 'dora', 'jurisdiction': 'eu', 'source_doc': 'L1_DORA_Regulation_2022_2554', 'doc_type': 'regulation', 'level': 1, 'article_num': 30, 'article_title': 'Key contractual provisions', 'section_ref': '', 'paragraph_idx': 0, 'topic_tags': 'incident_reporting|third_party_risk|resilience_testing|data_security|supervision', 'entity_scope': '', 'token_estimate': 1041}, excluded_embed_metadata_keys=['paragraph_idx', 'token_estimate'], excluded_llm_metadata_keys=['token_estimate'], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text="Article 30 — Key contractual provisions\n\nArticle 30\nKey contractual provisions\n1. The rights and obligations of the financial entity and of the ICT third-party service provider shall be clearly allocated\nand set out in writing. The full contract shall include the service level agreements and be documented in one written\ndo

In [26]:
# Cell 11 — Index health check

all_metadata = chroma_collection.get(include=['metadatas'])['metadatas']

print('=' * 65)
print('INDEX HEALTH CHECK')
print('=' * 65)
print(f'Vectors in index : {chroma_collection.count()}')
print(f'Expected chunks  : {len(raw_chunks)}')
status = '✓ MATCH' if chroma_collection.count() == len(raw_chunks) else '⚠ MISMATCH (duplicates removed or docs missing)'
print(f'Status           : {status}')

# By regulation
regs = Counter(m.get('regulation', 'unknown') for m in all_metadata)
print(f'\nVectors by regulation:')
for reg, cnt in sorted(regs.items()):
    print(f'  {reg:<14}: {cnt}')

# By jurisdiction
jurs = Counter(m.get('jurisdiction', 'unknown') for m in all_metadata)
print(f'\nVectors by jurisdiction:')
for jur, cnt in sorted(jurs.items()):
    print(f'  {jur:<10}: {cnt}')

# By level
levels = Counter(m.get('level', 0) for m in all_metadata)
print(f'\nVectors by level:')
for lv in sorted(levels):
    print(f'  Level {lv}: {levels[lv]} vectors')

# Per-document article coverage
print(f'\nPer-document article/section coverage:')
print(f'  {"Source doc":<48} {"Reg":<10} {"Items":>5} {"Range"}')
print('  ' + '─' * 75)

doc_articles = defaultdict(set)
doc_reg      = {}
for m in all_metadata:
    src = m.get('source_doc', 'unknown')
    art = m.get('article_num', -1)
    sec = m.get('section_ref', '')
    reg = m.get('regulation', '?')
    doc_reg[src] = reg
    if art and art != -1:
        doc_articles[src].add(('art', art))
    elif sec:
        doc_articles[src].add(('sec', sec))

for src in sorted(doc_articles.keys()):
    items = doc_articles[src]
    reg   = doc_reg.get(src, '?')
    arts  = sorted(i[1] for i in items if i[0] == 'art')
    secs  = sorted(str(i[1]) for i in items if i[0] == 'sec')
    if arts:
        range_str = f'Art {min(arts)}–{max(arts)}'
    elif secs:
        range_str = f'Sec {secs[0]}…{secs[-1]}'
    else:
        range_str = 'n/a'
    print(f'  {src[:48]:<48} {reg:<10} {len(items):>5}  {range_str}')

# Embedding dimension
sample_vec = chroma_collection.get(limit=1, include=['embeddings'])['embeddings']
if sample_vec is not None and len(sample_vec) > 0:
    print(f'\nEmbedding dimension: {len(sample_vec[0])} (expected 1024)')


INDEX HEALTH CHECK
Vectors in index : 798
Expected chunks  : 826
Status           : ⚠ MISMATCH (duplicates removed or docs missing)

Vectors by regulation:
  cross_reg     : 119
  dora          : 381
  nis2          : 179
  ps21_3        : 119

Vectors by jurisdiction:
  eu        : 615
  uk        : 183

Vectors by level:
  Level 1: 276 vectors
  Level 2: 440 vectors
  Level 3: 82 vectors

Per-document article/section coverage:
  Source doc                                       Reg        Items Range
  ───────────────────────────────────────────────────────────────────────────
  DORA_ESA_Guidelines_Simplified_ICT_Framework     cross_reg      9  Art 1–9
  DP3_22_Critical_Third_Parties_UK                 ps21_3         4  Sec 2.24…7.4
  EC_Guidelines_DORA_NIS2_Relationship_2023        cross_reg      1  Art 9–9
  FCA_FG22_5_Operational_Resilience                ps21_3        25  Sec 1.15…9.8
  L1_DORA_Directive_2022_2556                      dora          12  Art 1–18
  L1_DORA_Regulatio

In [27]:
# Cell 12 — Save updated index config
#
# Overwrites index_config.json with updated totals and regulation breakdown.
# All downstream notebooks (Phase 4, 5, 6, 8) read this file.

all_meta_config = chroma_collection.get(include=['metadatas'])['metadatas']

doc_types_config  = dict(Counter(m.get('doc_type',     'unknown') for m in all_meta_config))
levels_config     = dict(Counter(m.get('level',        0)         for m in all_meta_config))
regs_config       = dict(Counter(m.get('regulation',   'unknown') for m in all_meta_config))
jurs_config       = dict(Counter(m.get('jurisdiction', 'unknown') for m in all_meta_config))

index_config = {
    'created_at':        datetime.datetime.now().isoformat(),
    'embed_model':       EMBED_MODEL_NAME,
    'embed_dimension':   1024,
    'embed_batch_size':  EMBED_BATCH_SIZE,
    'similarity_metric': 'cosine',
    'device':            device,
    'chroma_dir':        str(CHROMA_DIR),
    'chroma_collection': CHROMA_COLLECTION,
    'total_vectors':     chroma_collection.count(),
    'source_chunks_file':str(ALL_CHUNKS_FILE),
    # Breakdowns
    'corpus_regulations':regs_config,
    'corpus_jurisdictions':jurs_config,
    'corpus_levels':     levels_config,
    'doc_types_indexed': doc_types_config,
    'notes': (
        'Multi-regulation corpus: DORA (EU) + NIS2 (EU) + PS21/3 (UK) + cross-regulation documents. '
        'BGE models require query_instruction prefix at retrieval time. '
        'topic_tags and entity_scope stored as pipe-separated strings. '
        'Filter fields: regulation, jurisdiction, level, doc_type, source_doc, article_num, section_ref.'
    )
}

CONFIG_FILE.write_text(json.dumps(index_config, indent=2), encoding='utf-8')
print(f'✓ Index config saved: {CONFIG_FILE}')
print()
print(json.dumps(index_config, indent=2))


✓ Index config saved: dora_rag/index_config.json

{
  "created_at": "2026-04-19T19:38:26.560343",
  "embed_model": "BAAI/bge-large-en-v1.5",
  "embed_dimension": 1024,
  "embed_batch_size": 16,
  "similarity_metric": "cosine",
  "device": "cpu",
  "chroma_dir": "dora_rag/dora_chroma_db",
  "chroma_collection": "dora_regulation",
  "total_vectors": 798,
  "source_chunks_file": "dora_rag/dora_chunks/all_chunks.json",
  "corpus_regulations": {
    "dora": 381,
    "nis2": 179,
    "ps21_3": 119,
    "cross_reg": 119
  },
  "corpus_jurisdictions": {
    "eu": 615,
    "uk": 183
  },
  "corpus_levels": {
    "1": 276,
    "2": 440,
    "3": 82
  },
  "doc_types_indexed": {
    "regulation": 114,
    "directive": 141,
    "RTS": 133,
    "ITS": 94,
    "delegated_regulation": 14,
    "guideline": 19,
    "statement": 3,
    "implementing_regulation": 37,
    "guidance": 42,
    "policy_statement": 85,
    "supervisory_statement": 16,
    "discussion_paper": 57,
    "Q_and_A": 43
  },
  "note

In [28]:
# Cell 13 — Phase 3b summary

print('\n' + '=' * 65)
print('PHASE 3b COMPLETE — MULTI-REGULATION EMBEDDING & INDEXING')
print('=' * 65)
print(f"""
Embedding model  : {EMBED_MODEL_NAME}
Vector dimension : 1024
Similarity       : cosine
Vector store     : ChromaDB (local persistent)
Collection       : {CHROMA_COLLECTION}
Total vectors    : {chroma_collection.count()}
Index location   : {CHROMA_DIR}

Corpus breakdown by regulation:
  DORA (EU)         : {regs_config.get('dora',      0)} vectors
  NIS2 (EU)         : {regs_config.get('nis2',      0)} vectors
  PS21/3 (UK)       : {regs_config.get('ps21_3',    0)} vectors
  Cross-regulation  : {regs_config.get('cross_reg', 0)} vectors

By jurisdiction:
  EU                : {jurs_config.get('eu',    0)} vectors
  UK                : {jurs_config.get('uk',    0)} vectors
  EU+UK             : {jurs_config.get('eu_uk', 0)} vectors

New filter dimensions available in Phase 4+:
  regulation    : dora | nis2 | ps21_3 | cross_reg
  jurisdiction  : eu | uk | eu_uk
  section_ref   : PS21/3 paragraph references

Outputs:
  {CHROMA_DIR}/    ← persistent vector index (all 3 regulations)
  {CONFIG_FILE}    ← updated index config for Phase 4b

Next step → open 04b_retrieval_multiregulation.ipynb
""")



PHASE 3b COMPLETE — MULTI-REGULATION EMBEDDING & INDEXING

Embedding model  : BAAI/bge-large-en-v1.5
Vector dimension : 1024
Similarity       : cosine
Vector store     : ChromaDB (local persistent)
Collection       : dora_regulation
Total vectors    : 798
Index location   : dora_rag/dora_chroma_db

Corpus breakdown by regulation:
  DORA (EU)         : 381 vectors
  NIS2 (EU)         : 179 vectors
  PS21/3 (UK)       : 119 vectors
  Cross-regulation  : 119 vectors

By jurisdiction:
  EU                : 615 vectors
  UK                : 183 vectors
  EU+UK             : 0 vectors

New filter dimensions available in Phase 4+:
  regulation    : dora | nis2 | ps21_3 | cross_reg
  jurisdiction  : eu | uk | eu_uk
  section_ref   : PS21/3 paragraph references

Outputs:
  dora_rag/dora_chroma_db/    ← persistent vector index (all 3 regulations)
  dora_rag/index_config.json    ← updated index config for Phase 4b

Next step → open 04b_retrieval_multiregulation.ipynb

